In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import json
import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 9})
mpl.rcParams['text.latex.preamble']= (
    r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}\usepackage{times}"
)
figurewidth_pt = 433.62
figurewidth_in = figurewidth_pt / 72.27

print(figurewidth_in)

In [ ]:
import jax
# jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", False)
import jax.numpy as jnp
import equinox as eqx 

In [ ]:
from admpe.utils.helper import pprint_helper
from admpe.evaluation.model_wrapping import get_wrapped_model_and_env
from admpe.data_management import (
    list_modeling_results, load_modeling_result, provide_modeling_results_overview, load_excitation_result, QUANTITATIVE_RESULTS_ROOT
)
from admpe.evaluation.modeling_evaluation import (
    evaluate_model, update_modeling_df, load_corresponding_excitation_result,
    evaluate_model_prediction, evaluate_model_rollout, evaluate_persistence_prediction
)
from admpe.evaluation.modeling_visualization import visualize_model_prediction, visualize_model_rollouts

In [ ]:
filter_testing_points = True
tag = "safely_excitable" if filter_testing_points else "full_box"

result_sources = {
    "gp_transition": ("submission_nov", 4),
    "neural_euler_large_1": ("submission_nov_1step", 3),
}

filter_testing_points = True
tag = "safely_excitable" if filter_testing_points else "full_box"
show_baseline = False

# TODO:

- show rollout performance comparison
- show one-step performance comparsion

In [ ]:
frames = []

for model_type, (exp_name, version) in result_sources.items():
    filename = (
        f"one-step-modeling-results-{exp_name}-{tag}-v{version}.parquet"
    )
    results = pd.read_parquet(QUANTITATIVE_RESULTS_ROOT / filename)
    results = results.loc[results["model_type"].eq(model_type)].copy()

    if results.empty:
        raise ValueError(f"No {model_type} results found in {filename}")

    frames.append(results)

modeling_results = pd.concat(frames, ignore_index=True)

USE_CURV64 = True

if USE_CURV64:
    new_sources = {
        "gp_transition": ("submission_nov_curv64", 4),
        "neural_euler_large_1": ("submission_nov_curv64_1step", 3),
    }
    curvature_objectives = [
        "dmpe_static_curv",
        "dmpe_static_mixture_curv",
    ]

    def is_replacement(df):
        return (
            df["system_name"].eq("pmsm_nominal")
            & df["excitation_objective"].isin(curvature_objectives)
        )

    new_frames = []

    for model_type, (experiment, version) in new_sources.items():
        filename = (
            f"one-step-modeling-results-{experiment}"
            f"-{tag}-v{version}.parquet"
        )
        new = pd.read_parquet(QUANTITATIVE_RESULTS_ROOT / filename)
        new = new.loc[
            new["model_type"].eq(model_type) & is_replacement(new)
        ].copy()

        expected = 10 if model_type == "gp_transition" else 50
        assert len(new) == expected, (
            f"{model_type}: expected {expected} results, found {len(new)}"
        )
        new_frames.append(new)

    modeling_results = pd.concat(
        [
            modeling_results.loc[~is_replacement(modeling_results)],
            *new_frames,
        ],
        ignore_index=True,
    )

modeling_results = modeling_results.loc[
    modeling_results["excitation_objective"].ne("dmpe_static_LOLA_ls")
].copy()

In [ ]:
from admpe.evaluation.modeling_visualization import visualize_model_prediction_metric_boxplots, visualize_model_rollout_metric_boxplots

In [ ]:
modeling_results

In [ ]:
def plot_one_step_prediction(df, metric="mse", plot_style="scatter", jitter=0.2):
    """Compare cached one-step errors: model rows by system columns."""
    if plot_style not in {"boxplot", "scatter"} or not 0 <= jitter < 0.5:
        raise ValueError("Choose boxplot/scatter and 0 <= jitter < 0.5.")
    metric = metric.lower()
    if metric not in {"mse", "rmse", "mae", "wce"}:
        raise ValueError("Choose mse, rmse, mae, or wce.")

    systems = ("fluid_tank", "pendulum", "cart_pole", "pmsm_nominal")
    titles = ("Fluid tank", "Pendulum", "Cart-pole", "PMSM")
    objectives = {
        "dmpe_uniform": "Uniform",
        "dmpe_static_dynamics": "Dynamics",
        "dmpe_static_mixture_dynamics": "Mixture dynamics",
        "dmpe_static_curv": "Curvature",
        "dmpe_static_mixture_curv": "Mixture curvature",
        # "dmpe_static_LOLA_ls": "LOLA",
    }
    column = f"prediction.metrics.{'rmse' if metric == 'mse' else metric}"
    columns = ["exp_id", "system_name", "model_type", "excitation_objective", column]
    missing = set(columns).difference(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    data = df.loc[df["system_name"].isin(systems), columns].copy()
    data = data.rename(columns={column: metric})
    if data.empty:
        raise ValueError("No one-step results selected.")
    if metric == "mse":
        data[metric] = data[metric] ** 2
    if not np.isfinite(data[metric]).all() or (data[metric] <= 0).any():
        raise ValueError("Log-scale errors must be finite and positive.")
    unknown = set(data["excitation_objective"]).difference(objectives)
    if unknown:
        raise ValueError(f"Unknown excitation objectives: {sorted(unknown)}")

    models = sorted(data["model_type"].unique())

    model_labels = {
        "gp_transition": "GP",
        "neural_euler_large_1": "MLP",
    }
    
    rng = np.random.default_rng(0)
    handles = {}
    fig, axes = plt.subplots(
        len(models), 4, figsize=(figurewidth_in, 0.8 * len(models) + 1.0),
        squeeze=False, sharex=True, sharey=False, layout="constrained",
    )
    for col, (system, title) in enumerate(zip(systems, titles)):
        axes[0, col].set_title(title, fontsize=9, y=1.0, pad=16)
        system_data = data.loc[data["system_name"].eq(system)]
        for row, model in enumerate(models):
            ax = axes[row, col]
            panel = system_data.loc[system_data["model_type"].eq(model)]
            ax.set_xticks([])
            ax.set_xlim(-0.6, len(objectives) - 0.4)
            if col == 0 and len(models) > 1:
                ax.set_ylabel(model_labels[model], fontsize=8)
            if panel.empty:
                ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                        ha="center", va="center", color="0.45", fontsize=8)
                ax.tick_params(axis="both", which="both", left=False, labelleft=False)
                continue
            for position, (objective, label) in enumerate(objectives.items()):
                values = panel.loc[panel["excitation_objective"].eq(objective), metric].to_numpy()
                if not values.size:
                    continue
                color = plt.get_cmap("tab10")(position)
                if plot_style == "scatter":
                    artist = ax.scatter(
                        position + rng.uniform(-jitter, jitter, values.size), values,
                        color=color, s=6, alpha=0.75, edgecolors="none", zorder=3,
                    )
                else:
                    boxes = ax.boxplot(
                        [values], positions=[position], widths=0.6,
                        patch_artist=True, manage_ticks=False,
                        boxprops={
                            "facecolor": color,
                            "edgecolor": color,
                            "alpha": 0.6,
                            "linewidth": 0.5,
                        },
                        medianprops={
                            "color": "black",
                            "linewidth": 0.5,
                        },
                        whiskerprops={"linewidth": 0.5},
                        capprops={"linewidth": 0.5},
                        flierprops={"marker": ".", "markeredgecolor": color},
                    )
                    artist = boxes["boxes"][0]
                artist.set_label(label)
                handles.setdefault(objective, artist)
            ax.tick_params(axis="y", labelsize=8)
            ax.grid(axis="y", which="major", alpha=0.3)
            ax.set_axisbelow(True)

        for ax in axes[:, col]:
            ax.set_yscale("log")
            ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(nbins=3))
            ax.yaxis.set_major_formatter(
                mpl.ticker.LogFormatterSciNotation(
                    labelOnlyBase=False,
                    minor_thresholds=(np.inf, np.inf),
                )
            )
            ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
            ax.yaxis.set_minor_formatter(
                mpl.ticker.NullFormatter()
            )

    fig.legend(
        handles=[handles[o] for o in objectives if o in handles],
        loc="outside lower center", title="Excitation objective",
        ncols=5, frameon=True, fontsize=8, title_fontsize=9,
        columnspacing=1.5, handletextpad=0.5,
    )
    fig.align_ylabels(axes[:, 0])
    fig.supylabel(f"One-step {metric.upper()}", fontsize=9)
    #fig.suptitle("One-step prediction performance", fontsize=9)
    return fig, axes, data


fig, axes, prediction_scores = plot_one_step_prediction(
    modeling_results, metric="mse", plot_style="scatter", jitter=0.2,
)
display(fig)
plt.close(fig)


In [ ]:
fig, axes, prediction_scores = plot_one_step_prediction(
    modeling_results,
    metric="mse",
    plot_style="boxplot",
    jitter=0.2,
)

from matplotlib.ticker import ScalarFormatter

for ax in axes.flat:
    formatter = ScalarFormatter(useMathText=True, useOffset=False)
    formatter.set_scientific(True)
    formatter.set_powerlimits((0, 0))

    ax.yaxis.set_major_formatter(formatter)
    ax.yaxis.get_offset_text().set_fontsize(8)

plt.savefig("paper_fig/quantitative_os_results.png", dpi=300, bbox_inches="tight")
plt.savefig("paper_fig/quantitative_os_results.pdf", bbox_inches="tight")
display(fig)
plt.close(fig)

In [ ]:
fig, axes, prediction_scores = plot_one_step_prediction(
    modeling_results,
    metric="mse",
    plot_style="scatter",
    jitter=0.2,
)

for ax in axes.flat:
    formatter = ScalarFormatter(useMathText=True, useOffset=False)
    formatter.set_scientific(True)
    formatter.set_powerlimits((0, 0))

    ax.yaxis.set_major_formatter(formatter)
    ax.yaxis.get_offset_text().set_fontsize(8)

plt.savefig("paper_fig/quantitative_os_results_scatter.png", dpi=300, bbox_inches="tight")
plt.savefig("paper_fig/quantitative_os_results_scatter.pdf", bbox_inches="tight")
display(fig)
plt.close(fig)